# 05 — DeepForest pretrained: inspeksi, inferensi, perbandingan dengan anotasi manusia

Baseline prototype: **kandidat tajuk pohon** dari `weecology/deepforest-tree` (citra pesawat NEON ±10 cm/px).
Bukan model tanaman pertanian. Inferensi memakai **adapter API yang sama** (`app.inference.DeepForestDetector`)
dan gambar terorientasi dari `app.imaging` — hasil di sini = hasil `/analyze`.

Prasyarat: `apps/api/scripts/download_deepforest.py` sudah dijalankan; `pip install deepforest==2.1.0 pandas==3.0.5`
di venv ini. Foto contoh: `samples/deepforest-neon/` (atribusi di `ATTRIBUTION.md`).

Metrik hanya dihitung bila ada **anotasi manusia** (XML). Angkanya berlaku untuk 3 foto contoh NEON ini saja —
**bukan** akurasi pada lahan pertanian atau foto K919.

In [ ]:
%matplotlib inline
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

import wallacea_ml
from wallacea_ml.prepare import ML_ROOT
print("wallacea_ml", wallacea_ml.__version__, "| ML_ROOT =", ML_ROOT)


def _df(rows):
    """Tabel: pandas bila terpasang (opsional), selain itu list apa adanya."""
    try:
        import pandas as pd
        return pd.DataFrame(rows)
    except ImportError:
        return rows


## 1. Foto & anotasi manusia

In [ ]:
from wallacea_ml import backend
from wallacea_ml.viz import draw_boxes
from wallacea_ml.voc import load_voc
SAMPLES = ML_ROOT.parent / "samples" / "deepforest-neon"
# Label XML: "Tree" (OSBS, YELL) dan "Alive"/"Dead" (SOAP_061: 9 hidup, 28 mati). Semuanya tajuk pohon → plant (eksplisit).
VOC_LABELS = {"Tree": "plant", "Alive": "plant", "Dead": "plant"}
photos = sorted(SAMPLES.glob("*.png"))
gts = {}
for p in photos:
    img = backend.load_oriented(p)
    voc = load_voc(p.with_suffix(".xml"), VOC_LABELS)
    assert (voc.width, voc.height) == img.size, "dimensi XML ≠ gambar terorientasi"
    gts[p.name] = voc
    print(f"{p.name}: {img.size[0]}×{img.size[1]} px, {len(voc.boxes)} kotak manusia {voc.raw_labels}")
fig, axes = plt.subplots(1, len(photos), figsize=(6 * len(photos), 6))
for ax, p in zip(axes, photos):
    ax.imshow(draw_boxes(backend.load_oriented(p), gts[p.name].boxes)); ax.set_title(f"{p.name} — anotasi manusia"); ax.axis("off")
plt.tight_layout(); plt.show()

## 2. Inferensi pretrained (adapter API, CPU) — waktu benar-benar diukur

In [ ]:
import time
PARAMS = dict(confidence=0.35, device="cpu", patch_size=400, patch_overlap=0.05)   # = default API
det = backend.build_api_deepforest(backend.default_deepforest_dir(), **PARAMS)
print("status:", det.info().state, "|", det.info().model_id, det.info().model_version)
t0 = time.perf_counter(); det.preload(); load_s = time.perf_counter() - t0
print(f"muat model: {load_s:.2f} s (sekali; tidak diulang per foto)")
preds, timing = {}, []
for p in photos:
    img = backend.load_oriented(p)
    t0 = time.perf_counter(); out = det.detect(img); dt_ = time.perf_counter() - t0
    preds[p.name] = out
    timing.append({"foto": p.name, "px": f"{img.size[0]}×{img.size[1]}", "mode": out.parameters["mode"],
                   "detik": round(dt_, 2), "kandidat": len(out.detections)})
display(_df(timing))
print("parameter:", {k: v for k, v in out.parameters.items()})

## 3. Perbandingan terhadap anotasi manusia (IoU ≥ 0.5)

In [ ]:
from wallacea_ml.evaluate import evaluate, match_details
data = {"model_id": det.info().model_id, "model_version": det.info().model_version, "split": "sampel",
        "domain": "public:neon-deepforest-samples", "iou_threshold": 0.5,
        "images": [{"image": n, "gt": [{"label": l, "bbox": b} for l, b in gts[n].boxes],
                    "pred": [{"label": d.label, "confidence": d.confidence, "bbox": d.bbox} for d in preds[n].detections]}
                   for n in gts]}
r = evaluate(data, leaks=[], conf_threshold=PARAMS["confidence"])
display(_df([{k: row[k] for k in ("image", "gt", "pred", "tp", "fp", "fn", "count_error")} for row in r.per_image]))
print({k: (round(getattr(r, k), 3) if isinstance(getattr(r, k), float) else getattr(r, k))
       for k in ("tp", "fp", "fn", "precision", "recall", "f1", "ap", "count_mae_per_image")})
for c in r.caveats: print("•", c)
print("• Hanya 3 foto contoh NEON (hutan AS). Bukan validasi untuk lahan pertanian / K919.")

## 4. False positive (merah) & objek terlewat (kuning) — dicatat per foto

In [ ]:
COLORS = {"tp": "plant", "fp": "gap", "fn": "check"}   # hijau = cocok, merah = FP, kuning = terlewat
for img_ in data["images"]:
    m = match_details(img_["gt"], [x for x in img_["pred"] if x["confidence"] >= PARAMS["confidence"]], 0.5, "plant")
    boxes = [("plant", d["bbox"]) for d, _ in m["tp"]] + [("gap", d["bbox"]) for d in m["fp"]] + [("check", g["bbox"]) for g in m["fn"]]
    print(f"{img_['image']}: {len(m['fp'])} false positive, {len(m['fn'])} terlewat; "
          f"FP confidence: {sorted(round(d['confidence'], 2) for d in m['fp'])[:10]}")
    plt.figure(figsize=(9, 9)); plt.imshow(draw_boxes(backend.load_oriented(SAMPLES / img_["image"]), boxes, max_side=1600))
    plt.title(f"{img_['image']} — hijau=cocok, merah=FP, kuning=terlewat"); plt.axis("off"); plt.show()

## 5. Foto K919

Belum ada foto K919 MAX di `ml/data/k919/raw/`. Tanpa foto dan anotasi manusia, **tidak ada** klaim kinerja
untuk K919. Bila ada foto, jalankan sel di bawah: hanya prediksi + waktu (tanpa metrik).

In [ ]:
K919 = ML_ROOT / "data" / "k919" / "raw"
ks = sorted(p for p in K919.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}) if K919.is_dir() else []
print(len(ks), "foto K919" if ks else "— belum ada foto K919 (hasil di atas hanya foto contoh NEON)")
for p in ks[:3]:
    img = backend.load_oriented(p)
    t0 = time.perf_counter(); out = det.detect(img); dt_ = time.perf_counter() - t0
    plt.figure(figsize=(12, 8)); plt.imshow(draw_boxes(img, [("pred", d.bbox) for d in out.detections], max_side=2400))
    plt.title(f"{p.name}: {len(out.detections)} kandidat tajuk pohon, {dt_:.1f} s — TANPA ground truth"); plt.axis("off"); plt.show()